%md
#### Assignment #4
#### Alia Kasem 
`All Foundation of empirical research`

In [0]:
# Load existing table into a DataFrame
flights = spark.table("default.flights")

In [0]:
from pyspark.sql.functions import col, avg, round
from pyspark.sql.window import Window

# 1. Remove rows with null arrival delays
flights_clean = flights.filter(col("arr_delay").isNotNull())

# 2. Create a window partitioned by carrier and month
carrier_month_window = Window.partitionBy("carrier", "month")

# 3. Add a column for the carrier's monthly average arrival delay
flights_with_avg = flights_clean.withColumn(
    "carrier_month_avg",
    avg("arr_delay").over(carrier_month_window)
)

# 4. Calculate how much worse each flight is than the carrier monthly average
flights_vs_avg = flights_with_avg.withColumn(
    "delay_difference",
    col("arr_delay") - col("carrier_month_avg")
)

# 5. Filter flights that performed worse than their carrier’s monthly average
worse_flights = flights_vs_avg.filter(col("delay_difference") > 0)

# 6. Select relevant columns and get top 20 worst offenders
top_20_worst = (
    worse_flights
    .select(
        "carrier",
        "month",
        "flight",
        "origin",
        "dest",
        "arr_delay",
        round("carrier_month_avg", 2).alias("carrier_month_avg"),
        round("delay_difference", 2).alias("delay_difference")
    )
    .orderBy(col("delay_difference").desc())
    .limit(20)
)

display(top_20_worst)

A window function allows us to calculate the carrier’s monthly average delay while retaining each individual flight record, enabling comparison between each flight and its group average. A groupBy operation would collapse the data, preventing row-level comparisons.

In [0]:
flights = spark.table("default.flights")

In [0]:
from pyspark.sql.functions import concat_ws

# Create a readable flight label
viz_df = top_20_worst.withColumn(
    "flight_label",
    concat_ws(" | ",
              top_20_worst.carrier,
              top_20_worst.flight.cast("string"),
              top_20_worst.origin,
              top_20_worst.dest)
)

display(viz_df.select("flight_label", "delay_difference"))

Databricks visualization. Run in Databricks to view.

I used a window function grouped by carrier and month to find each airline’s average arrival delay for the month, then compared each flight’s delay to that average. The data revealed some extreme outliers, such as a flight delayed by over 20 hours compared to its carrier’s usual performance. Several airlines, including DL, MQ, and AA, showed up in the top 20, which suggests that major delays happen across different carriers, not just one. This shows that while these severe delays are rare, they can be much worse than what airlines usually experience each month.

Outliers Per Carrier

In [0]:
flights = spark.table("default.flights")

In [0]:
from pyspark.sql.functions import col, avg
from pyspark.sql.window import Window

# Remove null arrival delays
flights_clean = flights.filter(col("arr_delay").isNotNull())

# Define window by carrier + month
carrier_month_window = Window.partitionBy("carrier", "month")

# Add monthly average delay
flights_with_avg = flights_clean.withColumn(
    "carrier_month_avg",
    avg("arr_delay").over(carrier_month_window)
)

# Calculate difference from monthly average
flights_vs_avg = flights_with_avg.withColumn(
    "delay_difference",
    col("arr_delay") - col("carrier_month_avg")
)

# Keep only worse-than-average flights
worse_flights = flights_vs_avg.filter(col("delay_difference") > 0)

In [0]:
from pyspark.sql.functions import count

carrier_outlier_counts = (
    worse_flights
    .groupBy("carrier")
    .agg(count("*").alias("num_extreme_outliers"))
    .orderBy(col("num_extreme_outliers").desc())
)

display(carrier_outlier_counts)

In [0]:
from pyspark.sql.functions import count

# Count how many worse-than-average flights each carrier has
carrier_outlier_counts = (
    worse_flights
    .groupBy("carrier")
    .agg(count("*").alias("num_extreme_outliers"))
    .orderBy(col("num_extreme_outliers").desc())
)

display(carrier_outlier_counts)

Databricks visualization. Run in Databricks to view.